# 3. Переразметка, свежие снимки и архив
Критерий согласованного отбора: BAD больше OK. Отрицательный ответ без бабочки — полноценная разметка с пустым списком рамок. Ручные решения имеют приоритет. История и оплата старых ответов сохраняются.

In [ ]:
from pathlib import Path
import sys, os, getpass, importlib
ROOT = Path.cwd()
if ROOT.name == 'clean_annotation':
    ROOT = ROOT.parent
CLEAN = ROOT / 'clean_annotation'
if not (CLEAN / 'workflow.py').exists():
    raise RuntimeError('Откройте ноутбук из корня проекта или clean_annotation')
if str(CLEAN) not in sys.path:
    sys.path.insert(0, str(CLEAN))
import workflow as w
importlib.reload(w)
PROFILE = 'existing'  # existing: текущие проекты; new: независимое воспроизведение
ENABLE_WRITES = False  # явное включение создания/изменений; запуск заданий отдельными ячейками
TOKEN = os.environ.get('YANDEX_TASKS_TOKEN') or getpass.getpass('OAuth-токен (скрытый ввод): ')
api = w.Client(TOKEN, write=ENABLE_WRITES)
del TOKEN


## Свежие ответы всех этапов
Скачайте перед выводами. Новый снимок фиксирует статусы на текущий момент, не исправляя ошибки ответов.

In [ ]:
DOWNLOAD = False
if DOWNLOAD:
    snapshot_folder = w.snapshot(api, PROFILE)
else:
    pointer = CLEAN / 'private/latest_snapshot.json'
    snapshot_folder = CLEAN / w.read(pointer)['directory'] if pointer.exists() else None
print(snapshot_folder)

## Анализ и переразметка
manual_by_photo.json фиксирует ваши ручные решения и финальные исправления. Для воспроизведения на новом датасете замените этот файл, а не переносите исторические ручные метки на другие фото. Последняя версия учитывается вместо старой; повторная неудача уже завершённого rework требует отдельного раунда, код не перезаписывает его незаметно.

In [ ]:
if snapshot_folder:
    decisions = w.review_decisions(snapshot_folder)
    from collections import Counter
    print(Counter(r['decision'] for r in decisions))

In [ ]:
SEND_REWORK = False
if SEND_REWORK:
    display(w.rework(api, PROFILE, snapshot_folder))

## Запуск и повторный REVIEW
Не открывайте пустой пул. После сдачи переразметки отправьте её в рабочий REVIEW — это новые версии боксов, им нужны собственные голоса.

In [ ]:
OPEN_REWORK = False
if OPEN_REWORK:
    display(w.launch(api, PROFILE, 'bbox_rework'))

In [ ]:
REVIEW_REWORK = False
if REVIEW_REWORK:
    display(w.handoff(api, PROFILE, 'bbox_rework', 'review_main'))

## Сохранить материалы и расходы для репозитория
export() работает без API, объединяет доступные исторические и свежие снимки, исключает дубликаты выдач и псевдонимизирует исполнителей. Это отчёт по статусам и reward, не банковская выписка. Комиссия 40% — явно указанное допущение; бонусы и возвраты не загружены.

In [ ]:
import evidence
importlib.reload(evidence)
budget = evidence.export()
display(budget)